# 📚 ACSM / DRM EPUBS to Kindle Converter

**Pure Python DRM Removal Tool for Personal EPUB Library Management**

This notebook removes Adobe DRM from your legally purchased EPUB files, allowing you to read them on any device including Kindle.

---

## ⚠️ Important Legal Notice

This tool is provided for **personal and educational purposes only**.

- ✅ **Legal Use**: Removing DRM from ebooks you have **legally purchased** for personal backup and device compatibility
- ✅ **Educational**: Understanding digital rights management and cryptographic concepts
- ❌ **Illegal Use**: Distributing, sharing, or pirating copyrighted content

**You are responsible for ensuring compliance with applicable laws in your jurisdiction.**

---

## 📋 Table of Contents

| Section | Description |
|---------|-------------|
| **1. Setup** | Install dependencies and configure paths |
| **2. Core Classes** | DRM decryption engine (run once) |
| **3. Convert Books** | 🚀 Main conversion — run this! |

---

## 🔄 Workflow

```
1. Purchase ebook → Download .acsm file
2. Open .acsm in Adobe Digital Editions → Downloads DRM-protected .epub
3. Copy .epub from ADE library to drm_epubs/ folder
4. Run this notebook → Removes DRM
5. Send to Kindle → Read anywhere!
```

---

## 📁 Directory Structure

```
📂 Project Folder
├── 📁 drm_epubs/       ← INPUT: Place DRM-protected EPUBs here (from ADE)
├── 📁 kindle_output/   ← OUTPUT: DRM-free EPUBs ready for Kindle
└── 📁 device_keys/     ← Auto-extracted ADE decryption keys
```

**ADE Library Locations:**
- **macOS**: `~/Documents/Digital Editions/`
- **Windows**: `Documents\My Digital Editions\`

---

## 📱 Sending to Kindle

After conversion, upload your DRM-free EPUBs via [amazon.com/sendtokindle](https://www.amazon.com/sendtokindle)

> 💡 Modern Kindles (2022+) natively support EPUB!

---
## 📦 Section 1: Setup
---

In [18]:
# ============================================================
# 1.1 Install Required Packages
# ============================================================

!pip install requests pycryptodome lxml rsa ebooklib beautifulsoup4 -q

print("✅ All packages installed successfully")

✅ All packages installed successfully


In [19]:
# ============================================================
# 1.2 Import Libraries & Configure Paths
# ============================================================

import os
import sys
import re
import base64
import shutil
import zipfile
import zlib
import xml.etree.ElementTree as ET
from pathlib import Path
from io import BytesIO

# Cryptography
from Crypto.Cipher import AES, PKCS1_v1_5
from Crypto.PublicKey import RSA

# EPUB handling
from ebooklib import epub
from bs4 import BeautifulSoup


class Config:
    """Configuration settings for the converter."""
    
    # Directory paths (relative to notebook location)
    DRM_EPUBS_DIR = "./drm_epubs"      # INPUT: DRM-protected EPUBs from ADE
    KINDLE_DIR = "./kindle_output"     # OUTPUT: DRM-free EPUBs for Kindle
    KEYS_DIR = "./device_keys"         # ADE decryption keys (auto-extracted)
    
    # Aliases for backward compatibility
    EPUB_DIR = DRM_EPUBS_DIR


# Create directories
for directory in [Config.DRM_EPUBS_DIR, Config.KINDLE_DIR, Config.KEYS_DIR]:
    Path(directory).mkdir(parents=True, exist_ok=True)

print("✅ Configuration complete")
print(f"""
📁 Directories:
   • INPUT  (DRM EPUBs):  {Config.DRM_EPUBS_DIR}
   • OUTPUT (Kindle):     {Config.KINDLE_DIR}
   • Keys:                {Config.KEYS_DIR}
""")

✅ Configuration complete

📁 Directories:
   • INPUT  (DRM EPUBs):  ./drm_epubs
   • OUTPUT (Kindle):     ./kindle_output
   • Keys:                ./device_keys



---
## ⚙️ Section 2: Core Classes
Run this section once to load all converter components.

---

In [20]:
# ============================================================
# 2.1 Adobe ADEPT DRM Decryptor
# ============================================================
# Core decryption engine using the correct Adobe ADEPT method

class AdobeADEPTDecryptor:
    """
    Adobe ADEPT DRM Decryption Engine.
    
    Extracts RSA private key from Adobe Digital Editions and uses it
    to decrypt DRM-protected EPUB files.
    
    Technical Details:
    - RSA key extracted from ADE's activation.dat (PKCS#8 DER format)
    - Content key decrypted using RSA PKCS1 v1.5
    - Files decrypted using AES-128-CBC with zero IV
    - First 16 bytes after decryption are skipped (original IV)
    - Content decompressed using zlib deflate (raw mode)
    """
    
    def __init__(self, key_dir="./device_keys"):
        self.key_dir = Path(key_dir)
        self.key_dir.mkdir(parents=True, exist_ok=True)
        self.rsa_key = None
        self._load_keys()
    
    def _load_keys(self):
        """Load RSA private key from ADE installation."""
        # Try loading from saved .der file first
        for der_file in self.key_dir.glob("*.der"):
            try:
                with open(der_file, 'rb') as f:
                    self.rsa_key = RSA.import_key(f.read())
                return True
            except Exception:
                pass
        
        # Try extracting from activation.dat
        activation_file = self.key_dir / "activation.dat"
        
        # Copy from ADE directory if not present
        if not activation_file.exists():
            ade_paths = {
                'darwin': Path.home() / "Library/Application Support/Adobe/Digital Editions/activation.dat",
                'win32': Path(os.environ.get('APPDATA', '')) / "Adobe/Digital Editions/activation.dat"
            }
            ade_path = ade_paths.get(sys.platform)
            if ade_path and ade_path.exists():
                shutil.copy2(ade_path, activation_file)
        
        if activation_file.exists():
            self.rsa_key = self._extract_key_from_activation(activation_file)
            if self.rsa_key:
                # Save as .der for faster loading next time
                der_path = self.key_dir / "adobe_key.der"
                with open(der_path, 'wb') as f:
                    f.write(self.rsa_key.export_key('DER'))
        
        return self.rsa_key is not None
    
    def _extract_key_from_activation(self, activation_file):
        """Extract RSA private key from activation.dat XML."""
        try:
            with open(activation_file, 'rb') as f:
                content = f.read().decode('utf-8', errors='ignore')
            
            # Find privateLicenseKey element (base64-encoded PKCS#8 DER)
            match = re.search(
                r'<(?:adept:)?privateLicenseKey[^>]*>([A-Za-z0-9+/=\s]+)</(?:adept:)?privateLicenseKey>',
                content
            )
            
            if match:
                key_b64 = match.group(1).replace('\n', '').replace(' ', '')
                key_der = base64.b64decode(key_b64)
                return RSA.import_key(key_der)
        except Exception:
            pass
        return None
    
    def decrypt_epub(self, input_path, output_path):
        """
        Decrypt a DRM-protected EPUB file.
        
        Args:
            input_path: Path to the DRM-protected EPUB
            output_path: Path for the decrypted output
            
        Returns:
            Path to decrypted file, or None if failed
        """
        if not self.rsa_key:
            return None
        
        try:
            with zipfile.ZipFile(input_path, 'r') as zin:
                file_list = zin.namelist()
                
                # Check for DRM markers
                if 'META-INF/rights.xml' not in file_list:
                    shutil.copy2(input_path, output_path)
                    return output_path
                
                # Decrypt content key from rights.xml
                rights_data = zin.read('META-INF/rights.xml')
                content_key = self._decrypt_content_key(rights_data)
                
                if not content_key:
                    return None
                
                # Get encrypted file list
                encrypted_files = set()
                if 'META-INF/encryption.xml' in file_list:
                    enc_xml = zin.read('META-INF/encryption.xml')
                    encrypted_files = self._parse_encryption_xml(enc_xml)
                
                # Create decrypted EPUB
                with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zout:
                    for item in zin.infolist():
                        # Skip DRM metadata files
                        if item.filename in ['META-INF/encryption.xml', 'META-INF/rights.xml']:
                            continue
                        
                        data = zin.read(item.filename)
                        
                        # Decrypt if encrypted
                        if item.filename in encrypted_files:
                            data = self._decrypt_file(data, content_key)
                        
                        zout.writestr(item, data)
                
                return output_path
                
        except Exception:
            return None
    
    def _decrypt_content_key(self, rights_xml):
        """Decrypt the AES content key using RSA."""
        try:
            content = rights_xml.decode('utf-8', errors='ignore')
            
            match = re.search(
                r'<(?:adept:)?encryptedKey[^>]*>([A-Za-z0-9+/=\s]+)</(?:adept:)?encryptedKey>',
                content
            )
            
            if not match:
                return None
            
            enc_key = base64.b64decode(match.group(1).replace('\n', '').replace(' ', ''))
            cipher = PKCS1_v1_5.new(self.rsa_key)
            sentinel = os.urandom(16)
            content_key = cipher.decrypt(enc_key, sentinel)
            
            if content_key == sentinel:
                return None
            
            return content_key[:16]  # AES-128 key
            
        except Exception:
            return None
    
    def _parse_encryption_xml(self, encryption_xml):
        """Parse encryption.xml to get list of encrypted files."""
        encrypted = set()
        try:
            root = ET.fromstring(encryption_xml)
            for elem in root.iter():
                if 'CipherReference' in elem.tag:
                    uri = elem.get('URI', '').lstrip('/')
                    if uri:
                        encrypted.add(uri)
        except Exception:
            pass
        return encrypted
    
    def _decrypt_file(self, data, key):
        """
        Decrypt a single file using Adobe ADEPT method.
        
        Adobe ADEPT uses:
        1. AES-128-CBC with zero IV
        2. First 16 bytes of result = original IV (skip)
        3. PKCS7 padding removal
        4. zlib deflate decompression
        """
        if len(data) < 16:
            return data
        
        try:
            # Pad to block size if needed
            if len(data) % 16 != 0:
                data = data + b'\x00' * (16 - len(data) % 16)
            
            # AES-CBC decrypt with zero IV
            cipher = AES.new(key, AES.MODE_CBC, b'\x00' * 16)
            decrypted = cipher.decrypt(data)
            
            # Skip first 16 bytes (original IV) 
            decrypted = decrypted[16:]
            
            # Remove PKCS7 padding
            if decrypted:
                pad = decrypted[-1]
                if 1 <= pad <= 16 and all(b == pad for b in decrypted[-pad:]):
                    decrypted = decrypted[:-pad]
            
            # Decompress (zlib raw deflate)
            try:
                dc = zlib.decompressobj(-15)
                decompressed = dc.decompress(decrypted)
                decompressed += dc.decompress(b'Z') + dc.flush()
                return decompressed
            except zlib.error:
                return decrypted  # Not compressed (fonts, images, etc.)
                
        except Exception:
            return data


print("✅ Adobe ADEPT Decryptor loaded")

✅ Adobe ADEPT Decryptor loaded


In [21]:
# ============================================================
# 2.2 Kindle EPUB Optimizer
# ============================================================
# Optimizes EPUB structure for best Kindle compatibility

class KindleOptimizer:
    """
    Optimize EPUB files for Kindle devices.
    
    Performs:
    - EPUB structure validation
    - HTML cleanup and optimization
    - Kindle-specific metadata injection
    - Clean ZIP packaging
    """
    
    @staticmethod
    def optimize(input_path, output_path):
        """
        Optimize an EPUB for Kindle.
        
        Args:
            input_path: Source EPUB path
            output_path: Optimized output path
            
        Returns:
            Path to optimized file, or None if failed
        """
        try:
            # Try using ebooklib for full optimization
            book = epub.read_epub(input_path)
            
            # Process HTML content
            for item in book.get_items():
                if item.get_type() == 9:  # XHTML
                    content = item.get_content()
                    if content:
                        try:
                            soup = BeautifulSoup(content, 'html.parser')
                            
                            # Remove Adobe DRM metadata tags
                            for meta in soup.find_all('meta', {'name': 'Adept.expected.resource'}):
                                meta.decompose()
                            
                            item.set_content(str(soup).encode('utf-8'))
                        except Exception:
                            pass
            
            # Write optimized EPUB
            epub.write_epub(output_path, book, {})
            return output_path
            
        except Exception:
            # Fallback: Create clean copy without ebooklib
            return KindleOptimizer._simple_copy(input_path, output_path)
    
    @staticmethod
    def _simple_copy(input_path, output_path):
        """Simple EPUB copy with basic cleanup."""
        try:
            with zipfile.ZipFile(input_path, 'r') as zin:
                with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zout:
                    for item in zin.infolist():
                        data = zin.read(item.filename)
                        
                        # Basic HTML cleanup for XHTML files
                        if item.filename.endswith(('.xhtml', '.html', '.htm')):
                            try:
                                content = data.decode('utf-8')
                                # Remove Adobe-specific metadata
                                content = re.sub(
                                    r'<meta[^>]*Adept\.expected\.resource[^>]*/?>', 
                                    '', 
                                    content
                                )
                                data = content.encode('utf-8')
                            except Exception:
                                pass
                        
                        zout.writestr(item, data)
            
            return output_path
        except Exception:
            return None


print("✅ Kindle Optimizer loaded")

✅ Kindle Optimizer loaded


In [ ]:
# ============================================================
# 2.3 ACSM to Kindle Converter (Main Class)
# ============================================================
# Complete end-to-end conversion pipeline

import tempfile

class ACSMToKindleConverter:
    """
    Complete EPUB DRM Removal and Kindle Converter.
    
    Handles the full conversion pipeline:
    1. Extract ADE decryption keys (automatic)
    2. Decrypt DRM-protected EPUBs
    3. Optimize for Kindle
    4. Output ready-to-use files
    
    Usage:
        converter = ACSMToKindleConverter()
        converter.convert_all()
    """
    
    def __init__(self, config=None):
        """Initialize converter with configuration."""
        self.config = config or Config
        self.decryptor = AdobeADEPTDecryptor(self.config.KEYS_DIR)
        self.optimizer = KindleOptimizer()
        self.has_keys = self.decryptor.rsa_key is not None
    
    def convert_file(self, input_path, output_name=None):
        """
        Convert a single DRM-protected EPUB to Kindle format.
        
        Args:
            input_path: Path to the DRM-protected EPUB
            output_name: Optional custom output name (without extension)
            
        Returns:
            Path to converted file, or None if failed
        """
        input_path = Path(input_path)
        
        if not input_path.exists():
            print(f"❌ File not found: {input_path}")
            return None
        
        if output_name is None:
            output_name = input_path.stem
        
        print(f"\n📖 Processing: {input_path.name}")
        
        # Step 1: Decrypt to temp file
        with tempfile.NamedTemporaryFile(suffix='.epub', delete=False) as tmp:
            temp_path = tmp.name
        
        print("   🔓 Removing DRM...")
        decrypted = self.decryptor.decrypt_epub(str(input_path), temp_path)
        
        if not decrypted:
            print("   ❌ Decryption failed")
            if os.path.exists(temp_path):
                os.remove(temp_path)
            return None
        print("   ✅ DRM removed")
        
        # Step 2: Optimize for Kindle
        kindle_path = Path(self.config.KINDLE_DIR) / f"{output_name}.epub"
        
        print("   📱 Optimizing for Kindle...")
        optimized = self.optimizer.optimize(temp_path, str(kindle_path))
        
        # Cleanup temp file
        if os.path.exists(temp_path):
            os.remove(temp_path)
        
        if optimized:
            print(f"   ✅ Ready: {kindle_path.name}")
            return str(kindle_path)
        else:
            print("   ⚠️  Optimization failed, using decrypted version")
            return None
    
    def convert_all(self, source_dir=None):
        """
        Convert all EPUB files in a directory.
        
        Args:
            source_dir: Directory containing EPUBs (default: Config.EPUB_DIR)
            
        Returns:
            Dictionary with 'success' and 'failed' lists
        """
        source_dir = Path(source_dir or self.config.EPUB_DIR)
        
        if not source_dir.exists():
            print(f"❌ Directory not found: {source_dir}")
            return {'success': [], 'failed': []}
        
        epub_files = list(source_dir.glob("*.epub"))
        
        if not epub_files:
            print(f"⚠️  No EPUB files found in: {source_dir}")
            return {'success': [], 'failed': []}
        
        print(f"\n📚 Found {len(epub_files)} EPUB file(s)")
        print("=" * 60)
        
        results = {'success': [], 'failed': []}
        
        for i, epub_path in enumerate(epub_files, 1):
            print(f"\n[{i}/{len(epub_files)}]")
            result = self.convert_file(epub_path)
            
            if result:
                results['success'].append(epub_path.name)
            else:
                results['failed'].append(epub_path.name)
        
        # Print summary
        self._print_summary(results)
        return results
    
    def _print_summary(self, results):
        """Print conversion summary."""
        print("\n" + "=" * 60)
        print("📊 CONVERSION SUMMARY")
        print("=" * 60)
        
        total = len(results['success']) + len(results['failed'])
        print(f"\n   Total:   {total} files")
        print(f"   ✅ Done:  {len(results['success'])}")
        print(f"   ❌ Failed: {len(results['failed'])}")
        
        if results['success']:
            print(f"\n📁 Output: {self.config.KINDLE_DIR}")
            for book in results['success']:
                print(f"   ✓ {book}")
        
        if results['failed']:
            print("\n⚠️  Failed files:")
            for book in results['failed']:
                print(f"   ✗ {book}")
        
        print("\n" + "=" * 60)
        print("📱 TO SEND TO KINDLE:")
        print("=" * 60)
        print("""
   Upload via https://www.amazon.com/sendtokindle
""")
    
    def check_status(self):
        """Check converter status and print diagnostic info."""
        print("🔍 CONVERTER STATUS")
        print("=" * 60)
        
        # Check keys
        if self.has_keys:
            print("✅ ADE decryption keys: Found")
        else:
            print("❌ ADE decryption keys: Not found")
            print("   → Run 'Extract ADE Keys' cell first")
        
        # Check directories
        for name, path in [
            ("EPUB input", self.config.EPUB_DIR),
            ("Kindle output", self.config.KINDLE_DIR),
            ("Keys", self.config.KEYS_DIR)
        ]:
            path = Path(path)
            exists = "✅" if path.exists() else "❌"
            files = len(list(path.glob("*"))) if path.exists() else 0
            print(f"{exists} {name}: {path} ({files} files)")
        
        # Check for EPUBs
        epub_dir = Path(self.config.EPUB_DIR)
        epubs = list(epub_dir.glob("*.epub"))
        if epubs:
            print(f"\n📚 EPUBs ready to convert:")
            for epub in epubs[:5]:
                print(f"   • {epub.name}")
            if len(epubs) > 5:
                print(f"   ... and {len(epubs) - 5} more")
        else:
            print(f"\n⚠️  No EPUBs found. Place files in: {epub_dir}")


print("✅ ACSM to Kindle Converter loaded")

✅ ACSM to Kindle Converter loaded


---
## 🚀 Section 3: Convert Books
Run these cells to convert your DRM-protected EPUBs to Kindle format.

---

In [23]:
# ============================================================
# 3.1 Extract ADE Keys (Run Once)
# ============================================================
# Extracts decryption keys from Adobe Digital Editions
# Only needs to be run once - keys are saved for future use

def extract_ade_keys():
    """Extract and save Adobe Digital Editions decryption keys."""
    print("🔑 EXTRACTING ADE KEYS")
    print("=" * 60)
    
    keys_dir = Path(Config.KEYS_DIR)
    
    # Check for existing keys
    existing_keys = list(keys_dir.glob("*.der"))
    if existing_keys:
        print(f"✅ Keys already extracted: {existing_keys[0].name}")
        return True
    
    # Find ADE directory
    ade_paths = {
        'darwin': Path.home() / "Library/Application Support/Adobe/Digital Editions",
        'win32': Path(os.environ.get('APPDATA', '')) / "Adobe/Digital Editions"
    }
    
    ade_dir = ade_paths.get(sys.platform)
    
    if not ade_dir or not ade_dir.exists():
        print("❌ Adobe Digital Editions not found")
        print(f"\n💡 Expected location: {ade_dir}")
        print("\nTo fix this:")
        print("   1. Install Adobe Digital Editions")
        print("   2. Sign in with your Adobe ID")
        print("   3. Download a book to authorize the device")
        print("   4. Run this cell again")
        return False
    
    print(f"📂 Found ADE: {ade_dir}")
    
    # Copy activation.dat
    activation_src = ade_dir / "activation.dat"
    if activation_src.exists():
        activation_dst = keys_dir / "activation.dat"
        shutil.copy2(activation_src, activation_dst)
        print(f"✅ Copied: activation.dat")
        
        # Extract RSA key
        decryptor = AdobeADEPTDecryptor(Config.KEYS_DIR)
        if decryptor.rsa_key:
            print(f"✅ RSA key extracted: {decryptor.rsa_key.size_in_bits()} bits")
            return True
    
    print("❌ Could not extract keys")
    return False


# Run key extraction
extract_ade_keys()

🔑 EXTRACTING ADE KEYS
📂 Found ADE: /Users/suman/Library/Application Support/Adobe/Digital Editions
✅ Copied: activation.dat
✅ RSA key extracted: 1024 bits


True

In [ ]:
# ============================================================
# 3.2 Convert All EPUBs (Main Conversion)
# ============================================================
# Converts all DRM-protected EPUBs in the drm_epubs folder

# Initialize converter
converter = ACSMToKindleConverter()

# Check status first
converter.check_status()

print("\n")

# Convert all EPUBs
results = converter.convert_all()

🔍 CONVERTER STATUS
✅ ADE decryption keys: Found
✅ EPUB input: drm_epubs (1 files)
✅ Kindle output: kindle_output (1 files)
✅ Keys: device_keys (2 files)

📚 EPUBs ready to convert:
   • Emotional Intelligence 2.0.epub



📚 Found 1 EPUB file(s)

[1/1]

📖 Processing: Emotional Intelligence 2.0.epub
   🔓 Removing DRM...
   ✅ DRM removed
   📱 Optimizing for Kindle...
   ✅ Ready: Emotional Intelligence 2.0.epub

📊 CONVERSION SUMMARY

   Total:   1 files
   ✅ Done:  1
   ❌ Failed: 0

📁 Output: ./kindle_output
   ✓ Emotional Intelligence 2.0.epub

📱 TO SEND TO KINDLE:

   Option 1: Send to Kindle Website
   → https://www.amazon.com/sendtokindle

   Option 2: Email to Kindle
   → Send EPUB to your @kindle.com address

   Option 3: USB Transfer
   → Copy EPUB to Kindle's 'documents' folder



In [ ]:
# ============================================================
# 3.3 Convert Single File (Optional Helper)
# ============================================================
# Use this to convert a specific file instead of batch processing

def convert_single(file_path: str) -> str:
    """
    Convert a single EPUB file.
    
    Args:
        file_path: Path to the EPUB file
        
    Returns:
        Path to the converted file, or None if failed
    """
    converter = ACSMToKindleConverter()
    return converter.convert_file(file_path)


# Example usage (uncomment to use):
# result = convert_single("./drm_epubs/my_book.epub")
# print(f"Output: {result}")

---
# 📚 Appendix
---

## 🔧 Troubleshooting

| Issue | Solution |
|-------|----------|
| **"No ADE keys found"** | Install Adobe Digital Editions, sign in, and download a book first |
| **"Decryption failed"** | Ensure the book was downloaded with the same Adobe ID |
| **"No EPUB files found"** | Place your `.epub` files in the `drm_epubs/` folder |
| **Book shows garbled text** | The DRM key doesn't match - re-download with your authorized ADE |

## 📋 How It Works

1. **Key Extraction**: Reads Adobe Digital Editions `activation.dat` to extract your RSA private key
2. **Content Key Decryption**: Uses RSA-PKCS1v1.5 to decrypt the AES content key from `rights.xml`
3. **File Decryption**: Decrypts content using AES-128-CBC with Adobe's ADEPT method
4. **Output**: Produces a standard, DRM-free EPUB readable on any device

## 🔐 Technical Details

Adobe ADEPT DRM encryption:
- **RSA-1024**: Encrypts per-book AES content key
- **AES-128-CBC**: Encrypts file contents (zero IV, skip first 16 bytes after decrypt)
- **PKCS7 padding**: Standard block cipher padding
- **zlib deflate**: Raw compression (wbits=-15)

---

## ⚖️ Legal Disclaimer

### Personal Use Only

This tool is provided **strictly for personal and educational purposes**.

**Permitted uses:**
- Removing DRM from ebooks you have **legally purchased** for personal backup
- Converting your own library for device compatibility (e.g., reading on Kindle)
- Educational study of digital rights management and cryptography

**Prohibited uses:**
- Distributing, sharing, or selling DRM-free copies
- Removing DRM from content you do not own
- Any use that violates copyright law

### Your Responsibility

By using this tool, you acknowledge that:
1. You are the legal owner or authorized user of the ebooks being processed
2. You will not distribute the resulting files
3. You are responsible for compliance with laws in your jurisdiction
4. The authors provide no warranty and assume no liability

### Legal Context

In many jurisdictions, removing DRM for personal backup of legally purchased content falls under fair use or similar provisions. However, laws vary by country. Consult local regulations if uncertain.

**References:**
- US: DMCA Section 1201 (with exemptions for personal use)
- EU: EUCD Article 6 (with member state variations)
- Fair use principles generally support format-shifting for personal use

---

*This tool is an educational project demonstrating cryptographic concepts. Use responsibly.*